# AI-Generated vs Real Image Detection: Formal Training Pipeline

This notebook runs the full project pipeline for detecting AI-generated images using a ResNet18 baseline and a ResNet18 + Efficient Channel Attention (ECA) model.

Pipeline:

1. Environment and GPU check
2. Load project code and dataset
3. Dataset quality inspection
4. Preprocessing and augmentation summary
5. Optional grid-search hyperparameter tuning
6. Final baseline and ECA training
7. Model comparison and statistical analysis
8. Visual explanation: FFT, error cases, and Grad-CAM
9. Package outputs for download


## 1. Runtime Check

Use a Kaggle GPU runtime for training. CPU is acceptable for dataset quality checks, but training will be much slower.

In [ ]:
import os
from pathlib import Path
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU. Enable GPU before full training.")

## 2. Clone or Update Project Repository

If the repository is already cloned in `/kaggle/working`, this cell updates it. Otherwise, it clones the GitHub repository.

In [ ]:
REPO_URL = "https://github.com/Irosha-Ranaweera/ai-image-detection-generalization.git"
PROJECT_DIR = Path("/kaggle/working/ai-image-detection-generalization")

if PROJECT_DIR.exists():
    %cd /kaggle/working/ai-image-detection-generalization
    !git pull
else:
    %cd /kaggle/working
    !git clone {REPO_URL}
    %cd /kaggle/working/ai-image-detection-generalization

print("Project directory:", Path.cwd())

## 3. Install Requirements

Kaggle usually includes PyTorch, torchvision, pandas, scikit-learn, matplotlib, and seaborn. This cell installs any missing project requirements.

In [ ]:
!pip install -q -r requirements.txt

## 4. Locate Dataset

Expected folder structure:

```text
ddata/
├── train/fake
├── train/real
├── val/fake
├── val/real
├── test/fake
└── test/real
```

The cell below automatically searches `/kaggle/input` for a folder named `ddata`.

In [ ]:
def find_dataset_dir():
    candidates = list(Path("/kaggle/input").rglob("ddata"))
    for candidate in candidates:
        required = [candidate / split for split in ["train", "val", "test"]]
        if all(path.exists() for path in required):
            return candidate
    raise FileNotFoundError("Could not find ddata with train/val/test inside /kaggle/input. Add your Kaggle dataset as notebook input.")

DATA_DIR = str(find_dataset_dir())
print("DATA_DIR =", DATA_DIR)
!find "$DATA_DIR" -maxdepth 2 -type d | sort | head -20

## 5. Experiment Control Panel

Set these flags depending on available GPU time.

- `RUN_DATA_QUALITY`: recommended, does not need GPU.
- `RUN_GRID_SEARCH`: formal hyperparameter tuning; can take time.
- `RUN_FINAL_TRAINING`: trains final baseline and ECA models.
- `RUN_COMPARISON`: compares checkpoints and saves analysis.
- `RUN_VISUALS`: creates FFT/error-case/Grad-CAM visualizations.

In [ ]:
RUN_DATA_QUALITY = True
RUN_GRID_SEARCH = False
RUN_FINAL_TRAINING = True
RUN_COMPARISON = True
RUN_VISUALS = True

RESULTS_DIR = "/kaggle/working/final_pipeline_outputs"
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

print("Results directory:", RESULTS_DIR)

## 6. Dataset Quality Inspection

This checks split counts, class counts, file types, corrupted images, image dimensions, and optionally duplicates.

Use the output tables and figures in the methodology section of the report.

In [ ]:
if RUN_DATA_QUALITY:
    os.environ["DATA_DIR"] = DATA_DIR
    os.environ["OUTPUT_DIR"] = f"{RESULTS_DIR}/dataset_quality"
    os.environ["CHECK_DUPLICATES"] = "false"
    !python scripts/check_dataset_quality.py
else:
    print("Skipped dataset quality check.")

## 7. Preprocessing and Augmentation Used

Training preprocessing:

- Resize to 224 × 224
- Random horizontal flip
- Random rotation
- Mild colour jitter
- Convert to tensor
- Normalize using ImageNet mean and standard deviation

Validation and test preprocessing:

- Resize to 224 × 224
- Convert to tensor
- Normalize using ImageNet mean and standard deviation

The optional FFT high-pass transform is used as an exploratory analysis of frequency-domain artifacts.

## 8. Optional Formal Grid Search

This performs controlled hyperparameter tuning. Hyperparameters are selected using validation performance. The test set is used only for final evaluation of the best validation trial.

Default grid:

- Models: baseline ResNet18, ResNet18 + ECA
- Learning rates: `1e-4`, `5e-5`
- Epochs: `5`, `10`
- Fine-tuning: layer4 enabled
- Batch size: `32`
- Input mode: RGB

In [ ]:
if RUN_GRID_SEARCH:
    os.environ["DATA_DIR"] = DATA_DIR
    os.environ["GRID_OUTPUT_DIR"] = f"{RESULTS_DIR}/grid_search_layer4"
    os.environ["NUM_WORKERS"] = "2"
    os.environ["SEED"] = "42"
    os.environ["GRID_MODELS"] = "baseline,eca"
    os.environ["GRID_MODEL_NAMES"] = "resnet18"
    os.environ["GRID_LEARNING_RATES"] = "1e-4,5e-5"
    os.environ["GRID_BATCH_SIZES"] = "32"
    os.environ["GRID_EPOCHS"] = "5,10"
    os.environ["GRID_FINE_TUNE_LAYER4"] = "true"
    os.environ["GRID_TRANSFORM_MODES"] = "rgb"
    os.environ["EARLY_STOPPING_PATIENCE"] = "3"
    os.environ["EVALUATE_TEST_EACH_TRIAL"] = "false"
    os.environ["GRID_SAVE_FIGURES"] = "true"
    !python scripts/grid_search.py
else:
    print("Skipped grid search. Set RUN_GRID_SEARCH = True to run formal tuning.")

## 9. Final Training: ResNet18 Baseline

This trains the baseline model with layer4 fine-tuning. The selected setup is based on previous experiments where layer4 fine-tuning substantially improved performance over classifier-only training.

In [ ]:
if RUN_FINAL_TRAINING:
    os.environ["DATA_DIR"] = DATA_DIR
    os.environ["MODEL_NAME"] = "resnet18"
    os.environ["EPOCHS"] = "5"
    os.environ["BATCH_SIZE"] = "32"
    os.environ["NUM_WORKERS"] = "2"
    os.environ["LEARNING_RATE"] = "5e-5"
    os.environ["FINE_TUNE_LAYER4"] = "true"
    os.environ["TRANSFORM_MODE"] = "rgb"
    os.environ["EARLY_STOPPING_PATIENCE"] = "0"
    os.environ["SAVE_PATH"] = f"{RESULTS_DIR}/checkpoints/best_resnet18_layer4_5ep.pth"
    os.environ["OUTPUT_DIR"] = f"{RESULTS_DIR}/figures_layer4_5ep"
    os.environ.pop("CHECKPOINT_PATH", None)
    !python scripts/train_baseline.py
else:
    print("Skipped final baseline training.")

## 10. Final Training: ResNet18 + ECA

This trains the improved model with Efficient Channel Attention. ECA adds lightweight channel-wise attention after the final convolutional block.

In [ ]:
if RUN_FINAL_TRAINING:
    os.environ["DATA_DIR"] = DATA_DIR
    os.environ["MODEL_NAME"] = "resnet18"
    os.environ["EPOCHS"] = "5"
    os.environ["BATCH_SIZE"] = "32"
    os.environ["NUM_WORKERS"] = "2"
    os.environ["LEARNING_RATE"] = "5e-5"
    os.environ["FINE_TUNE_LAYER4"] = "true"
    os.environ["TRANSFORM_MODE"] = "rgb"
    os.environ["EARLY_STOPPING_PATIENCE"] = "0"
    os.environ["SAVE_PATH"] = f"{RESULTS_DIR}/checkpoints/best_eca_resnet18_layer4_5ep.pth"
    os.environ["OUTPUT_DIR"] = f"{RESULTS_DIR}/figures_layer4_5ep"
    os.environ.pop("CHECKPOINT_PATH", None)
    !python scripts/train_attention.py
else:
    print("Skipped final ECA training.")

## 11. Compare Baseline vs ECA

This script evaluates both checkpoints on the same test set and saves:

- Model comparison summary
- Test predictions
- Confusion matrices
- ROC curve
- Precision-recall curve
- Probability density plot
- McNemar statistical test


In [ ]:
if RUN_COMPARISON:
    os.environ["DATA_DIR"] = DATA_DIR
    os.environ["MODEL_NAME"] = "resnet18"
    os.environ["BATCH_SIZE"] = "32"
    os.environ["NUM_WORKERS"] = "2"
    os.environ["LEARNING_RATE"] = "5e-5"
    os.environ["EPOCHS"] = "5"
    os.environ["TRANSFORM_MODE"] = "rgb"
    os.environ["FINE_TUNE_LAYER4"] = "true"
    os.environ["BASELINE_CHECKPOINT"] = f"{RESULTS_DIR}/checkpoints/best_resnet18_layer4_5ep.pth"
    os.environ["ECA_CHECKPOINT"] = f"{RESULTS_DIR}/checkpoints/best_eca_resnet18_layer4_5ep.pth"
    os.environ["ANALYSIS_DIR"] = f"{RESULTS_DIR}/analysis_layer4_5ep"
    !python scripts/compare_models.py
else:
    print("Skipped model comparison.")

## 12. FFT High-Pass Visual Examples

These images help explain whether high-frequency artifacts appear differently in fake and real images.

In [ ]:
if RUN_VISUALS:
    os.environ["DATA_DIR"] = DATA_DIR
    os.environ["OUTPUT_DIR"] = f"{RESULTS_DIR}/frequency_examples"
    !python scripts/visualize_frequency_examples.py
else:
    print("Skipped FFT visual examples.")

## 13. Error Case Visualizations

This creates image grids such as:

- Baseline wrong, ECA correct
- Baseline correct, ECA wrong
- Both models wrong

These are useful for qualitative error analysis in the presentation.

In [ ]:
if RUN_VISUALS:
    os.environ["DATA_DIR"] = DATA_DIR
    os.environ["PREDICTIONS_CSV"] = f"{RESULTS_DIR}/analysis_layer4_5ep/test_predictions.csv"
    os.environ["OUTPUT_DIR"] = f"{RESULTS_DIR}/error_cases_layer4_5ep"
    !python scripts/visualize_error_cases.py
else:
    print("Skipped error-case visualization.")

## 14. Grad-CAM Explanation

Grad-CAM highlights image regions that contributed most strongly to the model prediction. Use this to discuss whether the model focuses on meaningful facial/background regions or possible artifacts.

In [ ]:
if RUN_VISUALS:
    os.environ["DATA_DIR"] = DATA_DIR
    os.environ["MODEL_TYPE"] = "eca"
    os.environ["MODEL_NAME"] = "resnet18"
    os.environ["CHECKPOINT_PATH"] = f"{RESULTS_DIR}/checkpoints/best_eca_resnet18_layer4_5ep.pth"
    os.environ["OUTPUT_DIR"] = f"{RESULTS_DIR}/gradcam_eca_layer4_5ep"
    os.environ["FINE_TUNE_LAYER4"] = "true"
    os.environ["TRANSFORM_MODE"] = "rgb"
    !python scripts/visualize_gradcam.py
else:
    print("Skipped Grad-CAM visualization.")

## 15. Display Key Output Files

Use these generated images and files in the report/presentation.

In [ ]:
!find "$RESULTS_DIR" -maxdepth 3 -type f | sort | sed -n '1,120p'

## 16. Zip Outputs for Download

This creates one zip file containing checkpoints, figures, analysis files, and dataset quality outputs.

In [ ]:
ZIP_PATH = "/kaggle/working/final_pipeline_outputs.zip"
!cd /kaggle/working && zip -qr "$ZIP_PATH" final_pipeline_outputs
print("Saved:", ZIP_PATH)

## 17. Reporting Notes

Recommended final narrative:

- A pretrained ResNet18 baseline was used for binary classification.
- ECA attention was added as a lightweight architectural improvement.
- Experiments compared classifier-only training and layer4 fine-tuning.
- Hyperparameters were selected using validation results.
- The test set was used for final evaluation only.
- Confusion matrices, ROC-AUC, precision, recall, F1-score, McNemar test, FFT examples, error cases, and Grad-CAM were used for deeper analysis.

Important interpretation:

> ECA improved the frozen ResNet18 model and gave the best result in the 5-epoch layer4 fine-tuning experiment. Longer training did not always improve ECA, showing the importance of tuning and validation-based model selection.